# Treino do modelo de EPI (capacete + colete)

Cria a sua própria cópia do modelo `construction-site-safety/27`, treinada com **o mesmo dataset (versão 27, 2.801 imagens)**.
Dataset: *Construction Site Safety* por Roboflow Universe Projects — licença CC BY 4.0.

Rode as células **uma por vez, de cima para baixo** (botão ▶ à esquerda de cada uma).

Antes de começar: menu **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU → Salvar**.

In [ ]:
# PASSO 1 - Verifica a GPU e instala o YOLO e o Roboflow
import torch
print('GPU disponível:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NÃO! Ative a T4 GPU no menu e rode de novo.')
!pip install -q ultralytics roboflow

In [ ]:
# PASSO 2 - Baixa o dataset (versão 27) direto do Roboflow
# Vai aparecer uma caixa: cole sua chave do Roboflow (app.roboflow.com > Settings > API Keys) e aperte Enter.
# A chave não fica salva no notebook.
from getpass import getpass
from roboflow import Roboflow
import glob, os, yaml

rf = Roboflow(api_key=getpass('Chave do Roboflow: '))
ds = rf.workspace('roboflow-universe-projects').project('construction-site-safety').version(27).download('yolov11', location='/content/epi')

# Ajusta o data.yaml para os caminhos do Colab
data_yaml = glob.glob('/content/epi/**/data.yaml', recursive=True)[0]
raiz = os.path.dirname(data_yaml)
cfg = yaml.safe_load(open(data_yaml))
cfg['path'] = raiz
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images' if os.path.isdir(f'{raiz}/valid') else 'test/images'
cfg['test'] = 'test/images' if os.path.isdir(f'{raiz}/test') else cfg['val']
yaml.safe_dump(cfg, open(data_yaml, 'w'))

print('\nClasses do dataset:', cfg['names'])
for pasta in ('train', 'valid', 'test'):
    if os.path.isdir(f'{raiz}/{pasta}/images'):
        print(f'{pasta}: {len(os.listdir(f"{raiz}/{pasta}/images"))} imagens')
nomes = cfg['names'].values() if isinstance(cfg['names'], dict) else cfg['names']
faltam = [c for c in ('Hardhat', 'Safety Vest', 'Person') if c not in nomes]
print('OK, dataset compatível!' if not faltam else f'ATENÇÃO: não encontrei {faltam} - me avise os nomes das classes acima.')

In [ ]:
# PASSO 3 - Treina (uns 40 a 70 minutos). Pode deixar rodando, mas NÃO feche a aba.
# 'yolo11s' = mesmo porte do modelo original (YOLOv8s), só que da geração mais nova.
from ultralytics import YOLO
modelo = YOLO('yolo11s.pt')
modelo.train(data=data_yaml, epochs=50, imgsz=640, patience=15, project='/content/treino', name='epi', exist_ok=True)

In [ ]:
# PASSO 4 - Resultado: precisão por classe (olhe a coluna mAP50 de Hardhat, NO-Hardhat, Safety Vest, NO-Safety Vest, Person)
# Referência: o modelo original do Roboflow tem mAP50 = 0,84 no geral.
melhor = YOLO('/content/treino/epi/weights/best.pt')
metricas = melhor.val(data=data_yaml, split='test')
print(f'\n>>> mAP50 geral: {metricas.box.map50:.2f}')

# Mostra uma imagem de teste com as detecções
from IPython.display import Image, display
imagem_teste = sorted(glob.glob(f'{raiz}/test/images/*'))[0]
melhor.predict(imagem_teste, save=True, project='/content/teste', name='img', exist_ok=True)
display(Image(glob.glob('/content/teste/img/*')[0], width=700))

In [ ]:
# PASSO 5 - Baixa o modelo pronto (epi.pt) para o seu computador
import shutil
from google.colab import files
shutil.copy('/content/treino/epi/weights/best.pt', '/content/epi.pt')
files.download('/content/epi.pt')

## Pronto!
1. Coloque o `epi.pt` baixado em `monitor-epi\modelos\epi.pt`.
2. No `config.yaml`, troque `motor: "roboflow"` por `motor: "local"`.
3. Dê dois cliques em `iniciar_painel.bat`.